In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
print(torch.cuda.device_count())


x = torch.randn(3, 3, device="cuda")
y = x @ x
print(x)
print(y)



1
tensor([[-1.0247,  0.1211,  0.2004],
        [-0.7249, -1.5488, -0.9107],
        [-0.9341,  0.0832,  1.4232]], device='cuda:0')
tensor([[ 0.7750, -0.2949, -0.0304],
        [ 2.7162,  2.2353, -0.0309],
        [-0.4326, -0.1235,  1.7627]], device='cuda:0')


In [ ]:
import haiku as hk
import jax
import jax.numpy as jnp
from nucleotide_transformer.pretrained import get_pretrained_model

# Get pretrained model
parameters, forward_fn, tokenizer, config = get_pretrained_model(
    model_name="1B_agro_nt",
    embeddings_layers_to_save=(12,),
    max_positions=32,
)
forward_fn = hk.transform(forward_fn)


/home/htamm/miniconda3/envs/agront/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded model's hyperparameters.
Downloaded model's weights...


In [14]:
import random

nucleotides = ['A', 'C', 'G', 'T']

# --- generate 10 random sequences of length 6 ---
random.seed(0)  # one seed is enough for reproducibility
full_list_6_1000 = [
    ''.join(random.choice(nucleotides) for _ in range(6))
    for _ in range(2)
]

# --- create and append all middle-position variants ---
if full_list_6_1000:
    mid = len(full_list_6_1000[0]) // 2

    # work on a copy so we don't iterate over the growing list
    original_sequences = full_list_6_1000[:]

    # list comprehension = fast and compact
    variants = [
        seq[:mid] + nuc + seq[mid+1:]
        for seq in original_sequences
        for nuc in nucleotides
        # if you only want *changed* nucleotides in the middle, keep this:
        if nuc != seq[mid]
    ]

    # append all variants to the original list
    full_list_6_1000.extend(variants)

    full_list_6_1000.sort()

print(len(full_list_6_1000))
print(full_list_6_1000)


8
['GTGACG', 'GTGCCG', 'GTGGCG', 'GTGTCG', 'TTAATT', 'TTACTT', 'TTAGTT', 'TTATTT']


In [11]:
# Get data and tokenize it
sequences = full_list_6_1000
tokens_ids = [b[1] for b in tokenizer.batch_tokenize(sequences)]
tokens = jnp.asarray(tokens_ids, dtype=jnp.int32)

# Initialize random key
random_key = jax.random.PRNGKey(0)

# Inference
outs = forward_fn.apply(parameters, random_key, tokens)

# Get embeddings at layer 20
print(outs["embeddings_12"].shape)

W1202 12:42:29.282712 1350657 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W1202 12:42:29.289333 1329139 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


(10, 32, 1500)


In [3]:
print(outs["embeddings_12"])

[[[ -0.37829465  -7.817052     1.2317715  ...   2.403119     0.26740313
    -5.193776  ]
  [  3.6248162  -11.280587     9.808967   ...  -0.63659656  -5.446619
    -2.5782893 ]
  [  2.073442    -7.654725     6.6665025  ...   1.7031863    2.5718224
    -2.1831853 ]
  ...
  [ 14.380071     1.2837738   21.949232   ...  12.663685    10.645112
     2.3051465 ]
  [ 14.380071     1.2837738   21.949232   ...  12.663685    10.645112
     2.3051465 ]
  [ 14.380071     1.2837738   21.949232   ...  12.663685    10.645112
     2.3051465 ]]

 [[ -0.9202825   -6.171276     2.5364666  ...   3.795408     2.1836684
    -3.2639008 ]
  [  4.171793   -10.664946    12.369383   ...   1.0581734   -2.5833874
    -2.0383883 ]
  [  0.91910446  -5.6572685   12.832234   ...   4.4644156    3.8622031
    -2.9300437 ]
  ...
  [ 14.549945     1.5648042   22.035736   ...  12.752829    10.3577795
     2.566535  ]
  [ 14.549945     1.5648042   22.035736   ...  12.752829    10.3577795
     2.566535  ]
  [ 14.549945     1.5

In [ ]:
#!/usr/bin/env python3
"""
Einfaches Skript, um AgroNT-Embeddings für FASTA-Sequenzen zu berechnen
und deren Ähnlichkeit (Cosine Similarity) zu vergleichen.

Konfigurationen stehen direkt unten im Skript.
"""

from pathlib import Path

import haiku as hk
import jax
import jax.numpy as jnp
import numpy as np

from nucleotide_transformer.pretrained import get_pretrained_model


# ==========================
# Konfiguration
# ==========================

FASTA_DIR = Path("/home/htamm/models/agront/nucleotide-transformer/notebooks/fasta_files")
MODEL_NAME = "1B_agro_nt"
LAYER = 12           # letzter Layer (wie im Beispiel)
RNG_SEED = 0         # für reproduzierbare Ergebnisse


# ==========================
# Hilfsfunktionen
# ==========================

def read_fasta(path):
    """Sehr einfacher FASTA-Reader, gibt Liste von Sequenzen zurück."""
    seqs = []
    current = []
    with path.open() as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current:
                    seqs.append("".join(current).upper())
                    current = []
            else:
                current.append(line)
        if current:
            seqs.append("".join(current).upper())
    return seqs


def cosine_similarity_matrix(X):
    """Cosine-Similarity-Matrix für Vektoren in X (n, d)."""
    norms = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
    Xn = X / norms
    return Xn @ Xn.T  # (n, n)


# ==========================
# Hauptlogik
# ==========================

def main():
    # 1) FASTA-Dateien einlesen
    if not FASTA_DIR.is_dir():
        raise SystemExit(f"FASTA-Verzeichnis existiert nicht: {FASTA_DIR}")

    fasta_files = sorted(FASTA_DIR.glob("*.fa*"))
    if not fasta_files:
        raise SystemExit(f"Keine FASTA-Dateien in {FASTA_DIR} gefunden.")

    all_seqs_per_file = {}
    all_seqs_flat = []

    print(f"Suche FASTA-Dateien in {FASTA_DIR} ...")
    for ff in fasta_files:
        seqs = read_fasta(ff)
        if not seqs:
            print(f"Warnung: {ff.name} enthält keine Sequenzen, überspringe.")
            continue
        all_seqs_per_file[ff.name] = seqs
        all_seqs_flat.extend(seqs)

    if not all_seqs_per_file:
        raise SystemExit("Es wurden zwar FASTA-Dateien gefunden, aber keine Sequenzen.")

    # 2) maximale Sequenzlänge bestimmen
    max_len = max(len(s) for s in all_seqs_flat)
    print(f"Maximale Sequenzlänge über alle Dateien: {max_len}")

    # 3) Modell laden
    print(f"Lade Modell '{MODEL_NAME}' mit max_positions={max_len} ...")
    params, forward_fn, tokenizer, config = get_pretrained_model(
        model_name=MODEL_NAME,
        embeddings_layers_to_save=(LAYER,),
        max_positions=max_len,
    )
    forward_fn = hk.transform(forward_fn)

    # fixer Random Key
    rng = jax.random.PRNGKey(RNG_SEED)

    # 4) pro Datei Embeddings berechnen und Ähnlichkeit ausgeben
    for fname, seqs in all_seqs_per_file.items():
        print("\n" + "=" * 80)
        print(f"Datei: {fname}")
        print(f"  Anzahl Sequenzen: {len(seqs)}")
        print(f"  Sequenzlängen: {[len(s) for s in seqs]}")

        # Tokenisierung
        token_ids = [b[1] for b in tokenizer.batch_tokenize(seqs)]
        tokens = jnp.asarray(token_ids, dtype=jnp.int32)  # (batch, seq_len)

        # Inferenz
        outs = forward_fn.apply(params, rng, tokens)
        key = f"embeddings_{LAYER}"
        if key not in outs:
            raise KeyError(f"{key} nicht im Model-Output. Keys: {list(outs.keys())}")

        emb = np.array(outs[key])  # (n_seq, seq_len, hidden_dim)
        n_seq, seq_len, hidden_dim = emb.shape
        print(f"  Embeddings-Shape: (n_seq={n_seq}, seq_len={seq_len}, hidden_dim={hidden_dim})")

        # 4a) Mean-Pooling über die Sequenz
        mean_emb = emb.mean(axis=1)  # (n_seq, hidden_dim)

        # 4b) Embedding an der mittleren Position (SNP in der Mitte)
        mid_idx = seq_len // 2
        mid_emb = emb[:, mid_idx, :]  # (n_seq, hidden_dim)

        # 4c) Cosine-Similarity-Matrizen
        cos_mean = cosine_similarity_matrix(mean_emb)
        cos_mid = cosine_similarity_matrix(mid_emb)

        # Ausgabe (gerundet)
        print("  Cosine-Similarity (Mean-pooled):")
        print(np.round(cos_mean, 3))

        print("  Cosine-Similarity (Middle-Position):")
        print(np.round(cos_mid, 3))

        # Beispiel: erste zwei Sequenzen (falls vorhanden)
        if n_seq >= 2:
            def cos(u, v):
                return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12))

            print("  Beispiel (Seq 0 vs. Seq 1):")
            print(f"    Cos (Mean-pooled):      {cos(mean_emb[0], mean_emb[1]):.4f}")
            print(f"    Cos (Middle-Position):  {cos(mid_emb[0], mid_emb[1])::.4f}")

    print("\nFertig.")


if __name__ == "__main__":
    main()
